In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -q opencv-python mediapipe scikit-learn matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 15.4 MB/s eta 0:00:00


In [3]:
import cv2
import mediapipe as mp
import torch
import sklearn
import matplotlib

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("PyTorch:", torch.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Everything imported successfully ✅")

OpenCV: 5.0.0
MediaPipe: 1.0.1
PyTorch: 2.11.0+cu128
Scikit-learn: 1.6.1
Everything imported successfully ✅


In [4]:
from datasets import load_dataset

dataset = load_dataset("ai4bharat/INCLUDE")

print(dataset)

README.md:   0%|          | 0.00/4.91k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 78.2kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/val-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 13.1kB            

data/val-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3816 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/425 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1009 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['parent_label', 'label', 'video_path', 'include_50'],
        num_rows: 3816
    })
    val: Dataset({
        features: ['parent_label', 'label', 'video_path', 'include_50'],
        num_rows: 425
    })
    test: Dataset({
        features: ['parent_label', 'label', 'video_path', 'include_50'],
        num_rows: 1009
    })
})


In [5]:
labels = set()

for split in dataset:
    labels.update(dataset[split]["label"])

print("Total unique labels:", len(labels))

for label in sorted(labels):
    print(label)

Total unique labels: 263
1. Dog
1. Religion
1. loud
10. Energy
10. Mean
10. Plane
11. Car
11. War
11. rich
12. Peace
12. Truck
12. poor
13. Attack
13. Bicycle
13. thick
14. Bus
14. Election
14. thin
15. Boat
15. Newspaper
15. expensive
16. Gun
16. cheap
16. train ticket
17. Sport
17. Transportation
17. flat
18. City
18. Exercise
18. curved
19. Ball
19. House
19. male
2. Cat
2. Death
2. quiet
20. Price
20. Street or Road
20. female
21. Sign
21. Train Station
21. tight
22. Restaurant
22. Science
22. loose
23. Court
23. God
23. high
24. School
24. Table
24. low
25. Chair
25. Office
25. soft
26. Bed
26. University
26. hard
27. Dream
27. Park
27. deep
28. Store or Shop
28. Window
28. shallow
29. Door
29. Library
29. clean
3. Fish
3. Medicine
3. happy
30. Bedroom
30. Hospital
30. dirty
31. Kitchen
31. Temple
31. strong
32. Bathroom
32. Market
32. weak
33. India
33. Pencil
33. dead
34. Ground
34. Pen
34. alive
35. Bank
35. Photograph
35. heavy
36. Location
36. Soap
36. light
37. Book
37. Hat


In [8]:
# Find the actual labels corresponding to our target words

target_words = [
    "Doctor",
    "Patient",
    "Hospital",
    "Medicine",
    "sick",
    "healthy",
    "Today",
    "Tomorrow"
]

all_labels = set()

for split in dataset:
    all_labels.update(dataset[split]["label"])

for word in target_words:
    matches = [
        label for label in all_labels
        if label.split(". ", 1)[-1].strip().lower() == word.lower()
    ]

    print(f"{word:10} → {matches}")

Doctor     → ['87. Doctor']
Patient    → ['88. Patient']
Hospital   → ['30. Hospital']
Medicine   → ['3. Medicine']
sick       → ['98. sick']
healthy    → ['99. healthy']
Today      → ['73. Today']
Tomorrow   → ['74. Tomorrow']


In [9]:
# Get the exact dataset labels
selected_labels = []

for word in target_words:
    for label in all_labels:
        clean_label = label.split(". ", 1)[-1].strip()

        if clean_label.lower() == word.lower():
            selected_labels.append(label)

print("Selected labels:")
for label in sorted(selected_labels):
    print(label)

Selected labels:
3. Medicine
30. Hospital
73. Today
74. Tomorrow
87. Doctor
88. Patient
98. sick
99. healthy


In [10]:
selected_data = {}

for split in dataset:
    selected_data[split] = dataset[split].filter(
        lambda example: example["label"] in selected_labels
    )

print(selected_data)

Filter:   0%|          | 0/3816 [00:00<?, ? examples/s]

Filter:   0%|          | 0/425 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1009 [00:00<?, ? examples/s]

{'train': Dataset({
    features: ['parent_label', 'label', 'video_path', 'include_50'],
    num_rows: 99
}), 'val': Dataset({
    features: ['parent_label', 'label', 'video_path', 'include_50'],
    num_rows: 10
}), 'test': Dataset({
    features: ['parent_label', 'label', 'video_path', 'include_50'],
    num_rows: 23
})}


In [11]:
for split in selected_data:
    print(f"\n{split.upper()}")

    counts = {}

    for label in selected_data[split]["label"]:
        counts[label] = counts.get(label, 0) + 1

    for label, count in sorted(counts.items()):
        print(f"{label:20} : {count}")


TRAIN
3. Medicine          : 10
30. Hospital         : 15
73. Today            : 12
74. Tomorrow         : 12
87. Doctor           : 10
88. Patient          : 10
98. sick             : 15
99. healthy          : 15

VAL
3. Medicine          : 1
30. Hospital         : 1
73. Today            : 1
74. Tomorrow         : 1
87. Doctor           : 1
88. Patient          : 1
98. sick             : 2
99. healthy          : 2

TEST
3. Medicine          : 3
30. Hospital         : 4
73. Today            : 1
74. Tomorrow         : 1
87. Doctor           : 3
88. Patient          : 3
98. sick             : 4
99. healthy          : 4


In [12]:
print(dataset["train"].column_names)

['parent_label', 'label', 'video_path', 'include_50']


In [13]:
for split in selected_data:
    print(f"\n===== {split.upper()} =====")

    for i in range(min(10, len(selected_data[split]))):
        example = selected_data[split][i]

        print(
            f"{example['label']:20} → {example['video_path']}"
        )


===== TRAIN =====
74. Tomorrow         → Days_and_Time/74. Tomorrow/MVI_4619.MOV
73. Today            → Days_and_Time/73. Today/MVI_5471.MOV
98. sick             → Adjectives/98. sick/MVI_5253.MOV
98. sick             → Adjectives/98. sick/MVI_5171.MOV
74. Tomorrow         → Days_and_Time/74. Tomorrow/MVI_5035.MOV
99. healthy          → Adjectives/99. healthy/MVI_9286.MOV
87. Doctor           → Jobs/87. Doctor/MVI_5325.MOV
98. sick             → Adjectives/98. sick/MVI_9362.MOV
88. Patient          → Jobs/88. Patient/MVI_4760.MOV
98. sick             → Adjectives/98. sick/MVI_9442.MOV

===== VAL =====
98. sick             → Adjectives/98. sick/MVI_9444.MOV
73. Today            → Days_and_Time/73. Today/MVI_5031.MOV
3. Medicine          → Society/3. Medicine/MVI_8665.MP4
99. healthy          → Adjectives/99. healthy/MVI_5333.MOV
98. sick             → Adjectives/98. sick/MVI_5329.MOV
88. Patient          → Jobs/88. Patient/MVI_4475.MOV
30. Hospital         → Places/30. Hospital/MVI_356

In [14]:
print(selected_data["train"][0])

{'parent_label': 'Days_and_Time', 'label': '74. Tomorrow', 'video_path': 'Days_and_Time/74. Tomorrow/MVI_4619.MOV', 'include_50': False}


In [15]:
import requests

zenodo_api = "https://zenodo.org/api/records/4010759"

response = requests.get(zenodo_api)
response.raise_for_status()

zenodo = response.json()

print("Available files on Zenodo:\n")

for f in zenodo["files"]:
    print(f["key"])

Available files on Zenodo:

Adjectives_3of8.zip
Adjectives_4of8.zip
Adjectives_8of8.zip
Home_4of4.zip
Adjectives_5of8.zip
Adjectives_6of8.zip
Adjectives_7of8.zip
Pronouns_2of2.zip
Pronouns_1of2.zip
Society_2of3.zip
Places_4of4.zip
Places_3of4.zip
Society_1of3.zip
Seasons_1of1.zip
Places_2of4.zip
README.md
Places_1of4.zip
Society_3of3.zip
Days_and_Time_3of3.zip
People_5of5.zip
People_4of5.zip
People_3of5.zip
Days_and_Time_2of3.zip
Days_and_Time_1of3.zip
People_2of5.zip
Colours_2of2.zip
People_1of5.zip
Electronics_1of2.zip
Means_of_Transportation_2of2.zip
Means_of_Transportation_1of2.zip
Colours_1of2.zip
Electronics_2of2.zip
Animals_1of2.zip
Greetings_1of2.zip
Clothes_2of2.zip
Jobs_2of2.zip
Jobs_1of2.zip
Clothes_1of2.zip
Animals_2of2.zip
Greetings_2of2.zip
Home_1of4.zip
Home_2of4.zip
Home_3of4.zip
Adjectives_1of8.zip
Adjectives_2of8.zip
download_data.sh


In [18]:
import requests
import os
import time
from IPython.display import clear_output

# Zenodo record
zenodo_api = "https://zenodo.org/api/records/4010759"

response = requests.get(zenodo_api)
response.raise_for_status()
zenodo = response.json()

# Only the archives we need
required_archives = [
    "Jobs_1of2.zip",
    "Places_1of4.zip",
    "Society_3of3.zip",
    "Adjectives_8of8.zip",
    "Days_and_Time_3of3.zip"
]

download_dir = "/content/include_zips"
os.makedirs(download_dir, exist_ok=True)

# Create lookup table for Zenodo files
zenodo_files = {
    f["key"]: f["links"]["self"]
    for f in zenodo["files"]
}


def format_time(seconds):
    """Convert seconds into a readable time format."""
    if seconds <= 0:
        return "0s"

    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)

    if hours > 0:
        return f"{hours}h {minutes}m {seconds}s"
    elif minutes > 0:
        return f"{minutes}m {seconds}s"
    else:
        return f"{seconds}s"


# Download each archive
for file_index, filename in enumerate(required_archives, start=1):

    if filename not in zenodo_files:
        print(f"❌ Not found: {filename}")
        continue

    url = zenodo_files[filename]
    output_path = os.path.join(download_dir, filename)

    # Skip if already downloaded
    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        print(f"⏭️ {filename} already exists. Skipping.")
        continue

    print("\n" + "=" * 70)
    print(f"📦 File {file_index}/{len(required_archives)}: {filename}")
    print("=" * 70)

    start_time = time.time()

    with requests.get(url, stream=True) as r:
        r.raise_for_status()

        total = int(r.headers.get("content-length", 0))
        downloaded = 0

        with open(output_path, "wb") as f:

            for chunk in r.iter_content(chunk_size=1024 * 1024):

                if not chunk:
                    continue

                f.write(chunk)
                downloaded += len(chunk)

                # Calculate statistics
                elapsed = time.time() - start_time
                speed = downloaded / elapsed if elapsed > 0 else 0

                if total > 0:
                    percentage = downloaded / total * 100
                    remaining = (total - downloaded) / speed if speed > 0 else 0

                    downloaded_mb = downloaded / (1024 ** 2)
                    total_mb = total / (1024 ** 2)
                    speed_mb = speed / (1024 ** 2)

                    # Progress bar
                    bar_length = 40
                    filled = int(bar_length * percentage / 100)
                    bar = "█" * filled + "░" * (bar_length - filled)

                    clear_output(wait=True)

                    print("=" * 70)
                    print(f"📦 File {file_index}/{len(required_archives)}")
                    print(f"⬇️  {filename}")
                    print("=" * 70)

                    print(
                        f"[{bar}] {percentage:6.2f}%"
                    )

                    print(
                        f"Downloaded : {downloaded_mb:8.2f} MB / "
                        f"{total_mb:8.2f} MB"
                    )

                    print(
                        f"Speed      : {speed_mb:8.2f} MB/s"
                    )

                    print(
                        f"Elapsed    : {format_time(elapsed)}"
                    )

                    print(
                        f"Remaining  : {format_time(remaining)}"
                    )

                else:
                    # If Zenodo doesn't provide total file size
                    downloaded_mb = downloaded / (1024 ** 2)

                    clear_output(wait=True)

                    print("=" * 70)
                    print(f"📦 File {file_index}/{len(required_archives)}")
                    print(f"⬇️  {filename}")
                    print("=" * 70)
                    print(f"Downloaded: {downloaded_mb:.2f} MB")

    elapsed = time.time() - start_time
    final_size = os.path.getsize(output_path) / (1024 ** 2)

    print("\n" + "=" * 70)
    print(f"✅ COMPLETED: {filename}")
    print(f"Size      : {final_size:.2f} MB")
    print(f"Time      : {format_time(elapsed)}")
    print("=" * 70)


print("\n" + "=" * 70)
print("🎉 ALL REQUIRED ARCHIVES DOWNLOADED!")
print("=" * 70)

print("\nFiles:")
for filename in required_archives:

    path = os.path.join(download_dir, filename)

    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 ** 2)
        print(f"✅ {filename:30} {size:8.2f} MB")
    else:
        print(f"❌ {filename:30} NOT DOWNLOADED")

📦 File 2/5
⬇️  Places_1of4.zip
[█████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  13.69%
Downloaded :   182.00 MB /  1329.61 MB
Speed      :     0.65 MB/s
Elapsed    : 4m 39s
Remaining  : 29m 20s


KeyboardInterrupt: 